# Initialization

In [1]:
''' Initial imports. Keep collapsed to reduce clutter. '''
import sys
import os
import matplotlib.pyplot as plt
%matplotlib widget
import numpy as np
import gvar as gv

from os import path
from IPython.display import clear_output
from copy import deepcopy
import gc
import importlib
import subprocess
import json
import ipywidgets as widgets
from ipywidgets import interact
from ipywidgets import interact_manual
from IPython.display import display
import ast

In [2]:
NNType = ["deuteron", "dineutron"]

DeuteronIrreps = [
    [("0", "T1g", 0)], [('0', 'T1g', 1)],
    [('1', 'A2', 0)], [('1', 'A2', 1)], 
    [('1', 'E', 0)], [('1', 'E', 1)], [('4', 'E', 0)], [('4', 'E', 1)],
    [('2', 'A2', 0)], [('4', 'A2', 0)], [('4', 'A2', 1)], 
    [('2', 'B1', 0)], [('2', 'B2', 0)], [('2', 'B2', 3)],
    [('3', 'A2', 0)], [('3', 'A2', 1)], [('3', 'E', 0)]
    ]

DineutronIrreps = [ [('Not yet or maybe never in this notebook')] ]

# ''' The lines below will define the bounds for the interactive widget. '''

# MinNStates = 0
# MaxNStates = 5

# GEVPMinTime = 0
# GEVPMaxTime = 12

# ''' Max & Min time for N and R fit range '''
# NMinTime = 0
# NMaxTime = 20
# RMinTime = 0
# RMaxTime = 15

# MinBlock = 0
# MaxBlock = 10 



In [3]:
''' Interactive widget layouts. Feel free to change where needed. Keep collapsed to reduce clutter. '''
layout = widgets.Layout(width='450px')
style={'description_width': '200px'}

def WidgetTextToArray(widget):
    try:
        arr = ast.literal_eval(widget.value)
        return arr
    except Exception as e:
        print("Invalid format!")
        return None

# Fitting Procedure

### Directories and Stuff

In [4]:
import nn_fit as fit
p = dict()
p["debug"]   = False
p["verbose"] = False
p["latex"]   = True
p["save"] = True
p["fitter"] = 'scipy_least_squares'
p["fpath"] = {"nucleon": "./data/cosmon_c103_r005-8_nucleon.hdf5", 
                "nn": "./data/cosmon_c103_r005-8_deuteron_Swave.hdf5"}
p['get_Zj']         = True
# if 'deuteron' in p["fpath"]["nn"]:
#     p['Zjn_values'] = f"result/deuteron_Zjn_tNorm{p['t_norm']}_{p['gevp']}.h5"
# elif 'dineutron' in p["fpath"]["nn"]:
#     p['Zjn_values'] = f"result/dineutron_Zjn_tNorm{p['t_norm']}_{p['gevp']}.h5"
p['show_Zjn']       = False
p['do_gevp']        = False #set to True if you want to do gevp if it was already done and saved

## Fitting

### Sweep Fit
Sweep over multiple fits. Potentially takes hours. 

In [5]:
''' Interactive Functions. Feel free to change where needed. Keep collapsed to reduce clutter. '''
def SweepFitUpdateMain(TNorm, AgCons, NElastic, Ratio, Evp):
    p['t_norm']     = TNorm
    p["ratio"]      = Ratio
    p["version"]    = AgCons
    p['gevp']       = Evp
    p["r_n_el"]     = NElastic
    return p

def SweepFitUpdateBootStrap(BootStrap,NbsMax,Nbs,NbsSub,Bs0Width,BSPriors):
    p["bootstrap"]  = BootStrap
    p['Nbs_max']    = NbsMax
    p["nbs"]        = Nbs
    p["nbs_sub"]    = NbsSub
    p['bs0_width']  = Bs0Width
    p['bs_prior']   = BSPriors
    return p 

def SweepFitUpdateExtras(SvdStudy,SvdCut,Autotime,SigE0,SigEnn,PosZ,RatioType,IrrepType,GSConspire,RNInel,Ampi,AMN):
    p['svd_study']  = SvdStudy
    p['svdcut']     = SvdCut
    p["autotime"]   = Autotime
    p["sig_e0"]     = SigE0
    p["sig_enn"]    = SigEnn
    p["positive_z"] = PosZ
    p["ratio_type"] = RatioType
    p["gs_conspire"]= GSConspire
    p["r_n_inel"]   = RNInel
    p["ampi"]       = Ampi
    p["amn"]        = AMN
    p["irreps"]     = IrrepType
    p["dE_elastic"] = 2 * np.sqrt(p["amn"]**2 + 1 * (2 * np.pi / 48) ** 2) -2*p["amn"]
    return

def SweepingFitUpdate(Irrep,pt0,ptd,pblock,pnN,pN0,pNd,pR0,pRd):
    return

def SweepFitButton(b):
    if 'deuteron' in p["fpath"]["nn"]:
        p['Zjn_values'] = f"deuteron_Zjn_tNorm{p['t_norm']}_{p['gevp']}.h5"
    elif 'dineutron' in p["fpath"]["nn"]:
        p['Zjn_values'] = f"dineutron_Zjn_tNorm{p['t_norm']}_{p['gevp']}.h5"
    with SweepFitButtonOut:
        if SweepFitIrrep.value == 'all':
            Irrep   = DeuteronIrreps
        else:
            Irrep   = [SweepFitIrrep.value]
        pt0     = WidgetTextToArray(SweepFitPt0)
        ptd     = WidgetTextToArray(SweepFitPtd)
        pblock  = WidgetTextToArray(SweepFitPBlock)
        pnN     = WidgetTextToArray(SweepFitPnN)
        pN0     = WidgetTextToArray(SweepFitPN0)
        pNd     = WidgetTextToArray(SweepFitPNd)
        pR0     = WidgetTextToArray(SweepFitPR0)
        pRd     = WidgetTextToArray(SweepFitPRd)
        for h in range(len(Irrep)):
            for i in range(len(pt0)):
                for j in range(len(pblock)):
                    for k in range(len(pnN)): 
                        for l in range(len(pN0)):
                            for m in range(len(pR0)):
                                for n in range(len(pNd)):
                                    for o in range(len(pRd)):
                                        p["masterkey"]  = [Irrep[h]]
                                        p["t0"]     = pt0[i]
                                        p["td"]     = ptd[i]
                                        p["block"]  = pblock[j] 
                                        p['bs_seed']    = 'nn_c103_b%d' %p["block"]
                                        p["nstates"]    = pnN[k]
                                        p["trange"]     = {"N": [pN0[l], pNd[n]], "R": [pR0[m], pRd[o]]}
                                        print(f"Running fit: irrep={p['masterkey']} "
                                                f"t0={p['t0']} td={p['td']} "
                                                f"N={p['nstates']} block={p['block']} "
                                                f"trange={p["trange"]}"
                                                )

                                        gv.switch_gvar()

                                        params_json = json.dumps(p)

                                        result = subprocess.run(
                                            [sys.executable, "-u", "run_fit.py", params_json],
                                            capture_output=True,
                                            text=True
                                        )

                                        print("STDOUT:\n", result.stdout)
                                        print("STDERR:\n", result.stderr)

                                        if result.returncode != 0:
                                            raise RuntimeError("Fit crashed")
                                        
                                        gv.restore_gvar()

                                        plt.close("all")
                                        clear_output()
                                        gc.collect()
                        

        print("Sweep Fitting Completed")    
    return 


In [6]:
''' Interactive Widgets. Feel free to change where needed. Keep collapsed to reduce clutter. '''

''' ----- Sweep Portion -----'''
SweepIrreps             = deepcopy(DeuteronIrreps) + ['all']
SweepFitIrrep           = widgets.Dropdown(options=SweepIrreps, description='Choose 1 or choose all irreps', style=style, layout=layout)
SweepFitPt0             = widgets.Text(value='[4,4,5,5,6,6]', description='Start of GEVP times', style=style, layout=layout)
SweepFitPtd             = widgets.Text(value='[8,10,10,12,10,12]',description='End of GEVP times', style=style, layout=layout)
SweepFitPBlock          = widgets.Text(value='[8]',description='Block', style=style, layout=layout)
SweepFitPnN             = widgets.Text(value='[2,3,4]',description='N states', style=style, layout=layout)
SweepFitPN0             = widgets.Text(value='[3,4,5,7]',description='N Fit Start times', style=style, layout=layout)
SweepFitPNd             = widgets.Text(value='[20]',description='N Fit End times', style=style, layout=layout)
SweepFitPR0             = widgets.Text(value='[2,3,4,5,6,7,8,9,10,11]',description='R Fit Start times', style=style, layout=layout)
SweepFitPRd             = widgets.Text(value='[15]',description='R Fit End times', style=style, layout=layout)

SweepFitControls         = widgets.VBox([SweepFitIrrep,SweepFitPt0,SweepFitPtd,SweepFitPBlock,SweepFitPnN,SweepFitPN0,SweepFitPNd,SweepFitPR0,SweepFitPRd])

SweepFitInteractive = widgets.interactive_output(SweepingFitUpdate,
                                                            {   
                                                                "Irrep": SweepFitIrrep,
                                                                "pt0": SweepFitPt0,
                                                                "ptd": SweepFitPtd,
                                                                "pblock": SweepFitPBlock,
                                                                "pnN": SweepFitPnN,
                                                                "pN0": SweepFitPN0,
                                                                "pNd": SweepFitPNd,
                                                                "pR0": SweepFitPR0,
                                                                "pRd": SweepFitPRd
                                                            }
                                                        )

SweepFitParamsAccordion      = widgets.Accordion(children=[widgets.VBox([SweepFitControls, SweepFitInteractive])])
SweepFitParamsAccordion.set_title(0, "Sweep Fit Parameters")

''' ----- Main Portion ----- '''
SweepFitTNorm                = widgets.IntSlider(value=3, min=0, max=5, step=1, description='tNorm: ', style=style, layout=layout)
SweepFitNElastic             = widgets.IntSlider(value=0, min=0, max=5, step=1, description='R N Elastic states: ', style=style, layout=layout)
SweepFitAgCons               = widgets.Dropdown(options=['conspire', 'agnostic'], description='Agnostic or Conspiracy? ', orientation='horizontal', style=style, layout=layout)
SweepFitEvpGevp              = widgets.Dropdown(options=['evp','gevp'], description='evp or gevp', orientation='horizontal', style=style, layout=layout)
SweepFitRatio                = widgets.Checkbox(value=False, description='Ratio? ', style=style, layout=layout)

SweepFitMainControls         = widgets.VBox([SweepFitTNorm, SweepFitNElastic,SweepFitAgCons, SweepFitEvpGevp, SweepFitRatio])

SweepFitMainInteractive = widgets.interactive_output(SweepFitUpdateMain,
                                                            {   
                                                                "TNorm": SweepFitTNorm,
                                                                "AgCons": SweepFitAgCons,
                                                                "NElastic": SweepFitNElastic,
                                                                "Ratio": SweepFitRatio,
                                                                "Evp": SweepFitEvpGevp
                                                            }
                                                        )

SweepFitMainParamsAccordion      = widgets.Accordion(children=[widgets.VBox([SweepFitMainControls, SweepFitMainInteractive])])
SweepFitMainParamsAccordion.set_title(0, "Non-Sweeping Main Fit Parameters")

''' ----- Boot Strap Portion ----- '''
SweepFitBootStrap            = widgets.Checkbox(value=False, description='Bootstrap? ', style=style, layout=layout)
SweepFitNbsMax               = widgets.IntText(value=5000,description='Max amount of BS samples', style=style, layout=layout)
SweepFitNbs                  = widgets.IntText(value=5000,description='How many BS samples to do', style=style, layout=layout)
SweepFitNbsSub               = widgets.IntText(value=100,description='How many BS samples to do before saving results', style=style, layout=layout)
BS0SweepFitText              = widgets.HTML(value="During BS resampling, set the width of priors to 5*sigma where sigma is the gs uncertainty determined with boot0.  This is to stabilize BS resampling and help ensure the BS samples do not fall into a local minimum. ")
SweepFitBS0Width             = widgets.IntText(value=5,description='bs0 width', style=style, layout=layout)
SweepFitBSPrior              = widgets.Dropdown(options=['gs', 'all'], description='Randomize prior mean for gs or all priors', orientation='horizontal', style=style, layout=layout)

SweepFitBootStrapControls    = widgets.VBox([BS0SweepFitText, SweepFitBootStrap,SweepFitNbsMax,SweepFitNbs,SweepFitNbsSub,SweepFitBS0Width,SweepFitBSPrior])

SweepFitBootStrapInteractive = widgets.interactive_output(SweepFitUpdateBootStrap,
                                                            {   
                                                                "BootStrap": SweepFitBootStrap, 
                                                                "NbsMax": SweepFitNbsMax,
                                                                "Nbs": SweepFitNbs,
                                                                "NbsSub": SweepFitNbsSub,
                                                                "Bs0Width": SweepFitBS0Width,
                                                                "BSPriors": SweepFitBSPrior
                                                            }
                                                        )

SweepFitBootStrapParamsAccordion     = widgets.Accordion(children=[widgets.VBox([SweepFitBootStrapControls, SweepFitBootStrapInteractive])])
SweepFitBootStrapParamsAccordion.set_title(0, "Boot Strap Parameters")

''' ----- Extra Portion ----- ''' 
# Just the stuff that I don't know where to put for now. 
# If you would like, use the functions and widgets templates above to split this section up into what you find appropriate.
# Some of these are also not used anymore, so comment out or delete any that is unused to reduce clutter. 
# I have not found where irreps_ben is used except just existing in nn_parameters_base. 
SweepFitSvdStudy             = widgets.Checkbox(value=False, description='Study SVD? ', style=style, layout=layout)
SweepFitSvdCut               = widgets.FloatText(value=1e-8, description='SVD Cut', style=style, layout=layout)
SweepFitAutotime             = widgets.IntSlider(value=10, min=0, max= 20, step=1, description="Time used to estimate mean gs energy prior", style=style, layout=layout)
SweepFitSigE0                = widgets.FloatSlider(value=1, description="Multiplication factor for meff[autotime] for prior width for deltaE_gs", style=style, layout=layout)
SweepFitSigEnn               = widgets.FloatSlider(value=1, description="Multiplication factor for meff[autotime] for prior width for deltaE_nn", style=style, layout=layout)
SweepFitPosZ                 = widgets.Checkbox(value=True, description='Force overlaps to be positive? ', style=style, layout=layout)
SweepFitRatioType            = widgets.Dropdown(options=['data'], description='Ratio Type', orientation='horizontal', style=style, layout=layout)
SweepFitIrrepType            = widgets.Dropdown(options=["irreps_ben", "irreps"], description='Irreps type', orientation='horizontal', style=style, layout=layout)
SweepFitGSConspire           = widgets.Checkbox(value=False, description='Only add deltaE for ground state? ', style=style, layout=layout)
SweepFitRNInel               = widgets.IntSlider(value=2, min=0, max=10, step=1, description="R N Inelastic States", style=style, layout=layout)
SweepFitAmpi                 = widgets.FloatSlider(value=0.310810, description="Ampi is used to construct excited state gaps", style=style, layout=layout)
SweepFitAMN                  = widgets.FloatSlider(value=0.70262, description="Amn is used to estimate elastic excited state gaps", style=style, layout=layout)

# There is an extra dE_Elastic term that is in the function SweepFitUpdateExtras, if you are looking for it. 

SweepFitExtraControls        = widgets.VBox([SweepFitSvdStudy,SweepFitSvdCut,SweepFitAutotime,SweepFitSigE0,SweepFitSigEnn,SweepFitPosZ,SweepFitRatioType,SweepFitIrrepType,SweepFitGSConspire,SweepFitRNInel,SweepFitAmpi,SweepFitAMN])

SweepFitExtraInteractive     = widgets.interactive_output(SweepFitUpdateExtras,
                                                            {   
                                                                "SvdStudy": SweepFitSvdStudy,
                                                                "SvdCut": SweepFitSvdCut,
                                                                "Autotime": SweepFitAutotime,
                                                                "SigE0": SweepFitSigE0,
                                                                "SigEnn": SweepFitSigEnn,
                                                                "PosZ": SweepFitPosZ,
                                                                "RatioType": SweepFitRatioType,
                                                                "IrrepType": SweepFitIrrepType,
                                                                "GSConspire": SweepFitGSConspire,
                                                                "RNInel": SweepFitRNInel,
                                                                "Ampi": SweepFitAmpi,
                                                                "AMN": SweepFitAMN
                                                            }
                                                        )

SweepFitExtraParamsAccordion = widgets.Accordion(children=[widgets.VBox([SweepFitExtraControls, SweepFitExtraInteractive])])
SweepFitExtraParamsAccordion.set_title(0, "Extra Parameters")

''' ----- Button to Click ----- '''
SweepFitButtonOut            = widgets.Output()

''' ----- Accordion stack ----- '''
SweepFitAccordionStack       = widgets.VBox([SweepFitParamsAccordion, SweepFitMainParamsAccordion, SweepFitBootStrapParamsAccordion, SweepFitExtraParamsAccordion])

In [7]:
RunSweepFitButton           = widgets.Button(description='Run Sweep Fit')

RunSweepFitButton.on_click(SweepFitButton)

SweepFitTextHeader       = widgets.HTML(f"<b>Sweep over parameters to run multiple fits </b>")

display(SweepFitTextHeader, SweepFitAccordionStack, RunSweepFitButton, SweepFitButtonOut)

HTML(value='<b>Sweep over parameters to run multiple fits </b>')

Button(description='Run Sweep Fit', style=ButtonStyle())

Output()

### 1 Fit
Run only 1 specific fit

In [8]:
''' Interactive Functions. Feel free to change where needed. Keep collapsed to reduce clutter. '''
def FitUpdateMain(Irrep, GEVP,TNorm, NRange, RRange, NStates, AgCons, NElastic, Block, Ratio, Evp, Priors):
    p["masterkey"]  = [Irrep]
    p["t0"]         = GEVP[0]
    p["td"]         = GEVP[1]
    p['t_norm']     = TNorm
    p["ratio"]      = Ratio
    p["version"]    = AgCons
    p['gevp']       = Evp
    p["r_n_el"]     = NElastic
    p["block"]      = Block
    p['bs_seed']    = 'nn_c103_b%d' %p["block"]
    p["nstates"]    = NStates
    p["trange"]     = {"N": [NRange[0], NRange[1]], "R": [RRange[0], RRange[1]]}
    p["autopriors"] = Priors
    return p

def FitUpdateBootStrap(BootStrap,NbsMax,Nbs,NbsSub,Bs0Width,BSPriors):
    p["bootstrap"]  = BootStrap
    p['Nbs_max']    = NbsMax
    p["nbs"]        = Nbs
    p["nbs_sub"]    = NbsSub
    p['bs0_width']  = Bs0Width
    p['bs_prior']   = BSPriors
    return p 

def FitUpdateExtras(SvdStudy,SvdCut,Autotime,SigE0,SigEnn,PosZ,RatioType,IrrepType,GSConspire,RNInel,Ampi,AMN):
    p['svd_study']  = SvdStudy
    p['svdcut']     = SvdCut
    p["autotime"]   = Autotime
    p["sig_e0"]     = SigE0
    p["sig_enn"]    = SigEnn
    p["positive_z"] = PosZ
    p["ratio_type"] = RatioType
    p["gs_conspire"]= GSConspire
    p["r_n_inel"]   = RNInel
    p["ampi"]       = Ampi
    p["amn"]        = AMN
    p["irreps"]     = IrrepType
    p["dE_elastic"] = 2 * np.sqrt(p["amn"]**2 + 1 * (2 * np.pi / 48) ** 2) -2*p["amn"]
    return

def FitButton(b):
    if 'deuteron' in p["fpath"]["nn"]:
        p['Zjn_values'] = f"deuteron_Zjn_tNorm{p['t_norm']}_{p['gevp']}.h5"
    elif 'dineutron' in p["fpath"]["nn"]:
        p['Zjn_values'] = f"dineutron_Zjn_tNorm{p['t_norm']}_{p['gevp']}.h5"
    with FitButtonOut:
        clear_output()
        print(f"Running fit: irrep={p['masterkey']} "
                f"t0={p['t0']} td={p['td']} "
                f"N={p['nstates']} block={p['block']} "
                f"trange={p["trange"]}"
                )
        fit.main(p)
        plt.close('all')

        clear_output()
        
        print('Fitting completed.')
    return 

In [9]:
''' Interactive Widgets. Feel free to change where needed. Keep collapsed to reduce clutter. '''

''' ----- Main Portion ----- '''
FitIrrep                = widgets.Dropdown(options=DeuteronIrreps, description='Fit Irrep: ', style=style, layout=layout)
FitGEVPt0td             = widgets.IntRangeSlider(value=[3,8], min=0, max=12, step=1, description='GEVP t0-td: ', style=style, layout=layout)
FitTNorm                = widgets.IntSlider(value=3, min=0, max=5, step=1, description='tNorm: ', style=style, layout=layout)
FitNElastic             = widgets.IntSlider(value=0, min=0, max=5, step=1, description='R N Elastic states: ', style=style, layout=layout)
FitBlock                = widgets.IntSlider(value=2, min=0, max=10, step=1, description='Block: ', style=style, layout=layout)
FitNStates              = widgets.IntSlider(value=3, min=0, max=5, step=1, description='Nstates: ', style=style, layout=layout)
FitNRange               = widgets.IntRangeSlider(value=[3,20], min=0, max=20, step=1, description='N Range t0-td: ', style=style, layout=layout)
FitRRange               = widgets.IntRangeSlider(value=[3,15], min=0, max=15, step=1, description='R Range t0-td: ', style=style, layout=layout)
FitAgCons               = widgets.Dropdown(options=['conspire', 'agnostic'], description='Agnostic or Conspiracy? ', orientation='horizontal', style=style, layout=layout)
FitEvpGevp              = widgets.Dropdown(options=['evp','gevp'], description='evp or gevp', orientation='horizontal', style=style, layout=layout)
FitRatio                = widgets.Checkbox(value=False, description='Ratio? ', style=style, layout=layout)
FitPriors               = widgets.Dropdown(options=['auto','manual'], description='Auto or manual priors? ', style=style, layout=layout)

FitMainControls         = widgets.VBox([FitIrrep, FitGEVPt0td, FitTNorm, FitNStates, FitNElastic, FitNRange, FitRRange, FitBlock, FitAgCons, FitEvpGevp, FitRatio, FitPriors])

FitMainInteractive      = widgets.interactive_output(FitUpdateMain,
                                                            {   
                                                                "Irrep": FitIrrep,
                                                                "GEVP": FitGEVPt0td,
                                                                "TNorm": FitTNorm,
                                                                "NRange": FitNRange,
                                                                "RRange": FitRRange,
                                                                "NStates": FitNStates,
                                                                "AgCons": FitAgCons,
                                                                "NElastic": FitNElastic,
                                                                "Block": FitBlock,
                                                                "Ratio": FitRatio,
                                                                "Evp": FitEvpGevp,
                                                                "Priors": FitPriors
                                                            }
                                                        )

FitMainParamsAccordion      = widgets.Accordion(children=[widgets.VBox([FitMainControls, FitMainInteractive])])
FitMainParamsAccordion.set_title(0, "Main Fit Parameters")

''' ----- Boot Strap Portion ----- '''
FitBootStrap            = widgets.Checkbox(value=False, description='Bootstrap? ', style=style, layout=layout)
FitNbsMax               = widgets.IntText(value=5000,description='Max amount of BS samples', style=style, layout=layout)
FitNbs                  = widgets.IntText(value=5000,description='How many BS samples to do', style=style, layout=layout)
FitNbsSub               = widgets.IntText(value=100,description='How many BS samples to do before saving results', style=style, layout=layout)
BS0FitText              = widgets.HTML(value="During BS resampling, set the width of priors to 5*sigma where sigma is the gs uncertainty determined with boot0.  This is to stabilize BS resampling and help ensure the BS samples do not fall into a local minimum. ")
FitBS0Width             = widgets.IntText(value=5,description='bs0 width', style=style, layout=layout)
FitBSPrior              = widgets.Dropdown(options=['gs', 'all'], description='Randomize prior mean for gs or all priors', orientation='horizontal', style=style, layout=layout)

FitBootStrapControls    = widgets.VBox([BS0FitText, FitBootStrap,FitNbsMax,FitNbs,FitNbsSub,FitBS0Width,FitBSPrior])

FitBootStrapInteractive = widgets.interactive_output(FitUpdateBootStrap,
                                                            {   
                                                                "BootStrap": FitBootStrap, 
                                                                "NbsMax": FitNbsMax,
                                                                "Nbs": FitNbs,
                                                                "NbsSub": FitNbsSub,
                                                                "Bs0Width": FitBS0Width,
                                                                "BSPriors": FitBSPrior
                                                            }
                                                        )

FitBootStrapParamsAccordion     = widgets.Accordion(children=[widgets.VBox([FitBootStrapControls, FitBootStrapInteractive])])
FitBootStrapParamsAccordion.set_title(0, "Boot Strap Parameters")

''' ----- Extra Portion ----- ''' 
# Just the stuff that I don't know where to put for now. 
# If you would like, use the functions and widgets templates above to split this section up into what you find appropriate.
# Some of these are also not used anymore, so comment out or delete any that is unused to reduce clutter. 
# I have not found where irreps_ben is used except just existing in nn_parameters_base. 
FitSvdStudy             = widgets.Checkbox(value=False, description='Study SVD? ', style=style, layout=layout)
FitSvdCut               = widgets.FloatText(value=1e-8, description='SVD Cut', style=style, layout=layout)
FitAutotime             = widgets.IntSlider(value=10, min=0, max= 20, step=1, description="Time used to estimate mean gs energy prior", style=style, layout=layout)
FitSigE0                = widgets.FloatSlider(value=1, description="Multiplication factor for meff[autotime] for prior width for deltaE_gs", style=style, layout=layout)
FitSigEnn               = widgets.FloatSlider(value=1, description="Multiplication factor for meff[autotime] for prior width for deltaE_nn", style=style, layout=layout)
FitPosZ                 = widgets.Checkbox(value=True, description='Force overlaps to be positive? ', style=style, layout=layout)
FitRatioType            = widgets.Dropdown(options=['data'], description='Ratio Type', orientation='horizontal', style=style, layout=layout)
FitIrrepType            = widgets.Dropdown(options=["irreps_ben", "irreps"], description='Irreps type', orientation='horizontal', style=style, layout=layout)
FitGSConspire           = widgets.Checkbox(value=False, description='Only add deltaE for ground state? ', style=style, layout=layout)
FitRNInel               = widgets.IntSlider(value=2, min=0, max=10, step=1, description="R N Inelastic States", style=style, layout=layout)
FitAmpi                 = widgets.FloatSlider(value=0.310810, description="Ampi is used to construct excited state gaps", style=style, layout=layout)
FitAMN                  = widgets.FloatSlider(value=0.70262, description="Amn is used to estimate elastic excited state gaps", style=style, layout=layout)

# There is an extra dE_Elastic term that is in the function FitUpdateExtras, if you are looking for it. 

FitExtraControls        = widgets.VBox([FitSvdStudy,FitSvdCut,FitAutotime,FitSigE0,FitSigEnn,FitPosZ,FitRatioType,FitIrrepType,FitGSConspire,FitRNInel,FitAmpi,FitAMN])

FitExtraInteractive     = widgets.interactive_output(FitUpdateExtras,
                                                            {   
                                                                "SvdStudy": FitSvdStudy,
                                                                "SvdCut": FitSvdCut,
                                                                "Autotime": FitAutotime,
                                                                "SigE0": FitSigE0,
                                                                "SigEnn": FitSigEnn,
                                                                "PosZ": FitPosZ,
                                                                "RatioType": FitRatioType,
                                                                "IrrepType": FitIrrepType,
                                                                "GSConspire": FitGSConspire,
                                                                "RNInel": FitRNInel,
                                                                "Ampi": FitAmpi,
                                                                "AMN": FitAMN
                                                            }
                                                        )

FitExtraParamsAccordion = widgets.Accordion(children=[widgets.VBox([FitExtraControls, FitExtraInteractive])])
FitExtraParamsAccordion.set_title(0, "Extra Parameters")

''' ----- Button to Click ----- '''
FitButtonOut            = widgets.Output()

''' ----- Accordion stack ----- '''
FitAccordionStack       = widgets.VBox([FitMainParamsAccordion, FitBootStrapParamsAccordion, FitExtraParamsAccordion])

In [10]:
RunFitButton           = widgets.Button(description='Run Fit')

RunFitButton.on_click(FitButton)

OneFitTextHeader       = widgets.HTML(f"<b>Run 1 specific fit </b>")

display(OneFitTextHeader, FitAccordionStack, RunFitButton, FitButtonOut)

HTML(value='<b>Run 1 specific fit </b>')

Button(description='Run Fit', style=ButtonStyle())

Output()

# Plotting

### 1 Plot

In [11]:
''' Interactive Functions. Feel free to change where needed. Keep collapsed to reduce clutter. '''
import plot_nn_stability_gevp as plotgevp

def Plot(filename,irrep,lims,PlotnNList,PlotnnElList,PlotGEVP0List,PlotGEVPdList,PlottminList,plotfigtype,plotratio,plotevp,plotgscons):
    argv = [
        "plot_nn_stability_gevp.py",
        filename
    ] 
    GEVPt0       = WidgetTextToArray(PlotGEVP0List)
    GEVPtd       = WidgetTextToArray(PlotGEVPdList)
    PlotGEVPList= []
    for i in range(len(GEVPt0)):
        PlotGEVPList.append(f'{GEVPt0[i]}-{GEVPtd[i]}')
    argv += ["--n_N"] + [str(x) for x in WidgetTextToArray(PlotnNList)]
    argv += ["--nn_el"] + [str(x) for x in WidgetTextToArray(PlotnnElList)]
    argv += ["--gevp"] + [str(x) for x in PlotGEVPList]
    argv += ["--tmin"] + [str(x) for x in WidgetTextToArray(PlottminList)]
    argv += ["--fig_type", plotfigtype]
    if plotratio:
        argv.append("--ratio")
    if not plotevp:
        argv.append("--evp")
    if plotgscons:
        argv.append("--gs_cons")
    sys.argv = argv
    plt.close('all')
    plotgevp.main(params=irrep)
    clear_output()
    fig     = plt.gcf()
    TopAx   = fig.axes[0]
    MidAx   = fig.axes[1]
    if (lims[0]-lims[1])!=0:
        TopAx.set_ylim(lims[1],lims[0])
    if (lims[3]-lims[2])!=0:
        MidAx.set_ylim(lims[3],lims[2])
    plt.show()

def PlotMakeFilename(Irrep, GEVP, TNorm, NRange, RRange, NStates, AgCons, NElastic, Block, Ratio, SVD, SVDCut):
    nn_iso  = 'deuteron'
    irrep   = Irrep[0]
    RepPath = f"{irrep[0]}{irrep[1]}{irrep[2]}"
    ft0td   = f'{GEVP[0]}-{GEVP[1]}'
    fN      = f'{NRange[0]}-{NRange[1]}'
    fagcons = AgCons
    fnN     = NStates
    fe      = NElastic
    fR      = f'{RRange[0]}-{RRange[1]}'
    fRatio  = Ratio
    fBlock  = Block
    ftnorm  = TNorm
    if SVD:
        fSVDCut = SVDCut
        filename = f"result/{RepPath}/NN_{nn_iso}_tnorm{ftnorm}_t0-td_{ft0td}_N_n{fnN}_t_{fN}_NN_{fagcons}_e{fe}_t_{fR}_ratio_{fRatio}_block{fBlock}_svdcut{fSVDCut}.pickle"
    else:
        filename = f"result/{RepPath}/NN_{nn_iso}_tnorm{ftnorm}_t0-td_{ft0td}_N_n{fnN}_t_{fN}_NN_{fagcons}_e{fe}_t_{fR}_ratio_{fRatio}_block{fBlock}.pickle"
    return filename

def PlottingButton(b):
    with PlotButtonOut:
        plt.close('all')
        clear_output()
        PlotFileName= PlotMakeFilename(Irrep=PlotIrrep.value, GEVP=PlotGEVPt0td.value, 
                                   TNorm=PlotTNorm.value, NRange=PlotNRange.value, RRange=PlotRRange.value, 
                                   NStates=PlotNStates.value, AgCons=PlotAgCons.value, 
                                   NElastic=PlotNElastic.value, Block=PlotBlock.value, Ratio=PlotRatio.value, SVD=PlotSvdStudy.value, SVDCut=PlotSvdCut.value)
        limits      = [Plot0UpLim.value,Plot0LowLim.value,Plot1UpLim.value,Plot1LowLim.value]
        Plot(filename=PlotFileName,irrep=PlotIrrep.value, lims=limits,PlotnNList=PlotArgsNN,PlotnnElList=PlotArgsNNEL,PlotGEVP0List=PlotArgsGEVP0,PlotGEVPdList=PlotArgsGEVPd,PlottminList=PlotArgsTmin,plotfigtype=PlotArgsFig.value,plotratio=PlotArgsRatio.value,plotevp=PlotArgsEvp.value,plotgscons=PlotArgsGSCons.value)

def PlotUpdate(Irrep, GEVP, TNorm, NRange, RRange, NStates, AgCons, NElastic, Block, Ratio, SVD, SVDCut):
    return

def PlotMakeArgs(PlotnNList,PlotnnElList,PlotGEVP0List,PlotGEVPdList,PlottminList,plotfigtype,plotratio,plotevp,plotgscons):
    return

In [12]:
''' Interactive Widgets. Feel free to change where needed. Keep collapsed to reduce clutter. '''
PlotIrrep                 = widgets.Dropdown(options=DeuteronIrreps, description='Choose Irrep: ', style=style, layout=layout)
PlotGEVPt0td              = widgets.IntRangeSlider(value=[3,8], min=0, max=12, step=1, description='GEVP t0-td: ', style=style, layout=layout)
PlotTNorm                 = widgets.IntSlider(value=3, min=0, max=5, step=1, description='tNorm: ', style=style, layout=layout)
PlotNElastic              = widgets.IntSlider(value=0, min=0, max=5, step=1, description='R N Elastic states: ', style=style, layout=layout)
PlotBlock                 = widgets.IntSlider(value=2, min=0, max=10, step=1, description='Block: ', style=style, layout=layout)
PlotNStates               = widgets.IntSlider(value=3, min=0, max=5, step=1, description='Nstates: ', style=style, layout=layout)
PlotNRange                = widgets.IntRangeSlider(value=[3,20], min=0, max=20, step=1, description='N Range t0-td: ', style=style, layout=layout)
PlotRRange                = widgets.IntRangeSlider(value=[3,15], min=0, max=15, step=1, description='R Range t0-td: ', style=style, layout=layout)
PlotAgCons                = widgets.Dropdown(options=['conspire', 'agnostic'], description='Agnostic or Conspiracy? ', orientation='horizontal', style=style, layout=layout)
PlotSvdStudy              = widgets.Checkbox(value=False, description='Study SVD? ', style=style, layout=layout)
PlotSvdCut                = widgets.Text(value='1e-08', description='SVD Cut', style=style, layout=layout)

PlotRatio                 = widgets.Checkbox(value=False, description='Ratio? ', style=style, layout=layout)

PlotControls              = widgets.VBox([PlotIrrep, PlotGEVPt0td, PlotTNorm, PlotNStates, PlotNElastic, PlotNRange, PlotRRange, PlotBlock, PlotAgCons, PlotRatio, PlotSvdStudy, PlotSvdCut])

PlotInteractive           = widgets.interactive_output(PlotUpdate,
                                                            {   
                                                                "Irrep": PlotIrrep,
                                                                "GEVP": PlotGEVPt0td,
                                                                "TNorm": PlotTNorm,
                                                                "NRange": PlotNRange,
                                                                "RRange": PlotRRange,
                                                                "NStates": PlotNStates,
                                                                "AgCons": PlotAgCons,
                                                                "NElastic": PlotNElastic,
                                                                "Block": PlotBlock,
                                                                "Ratio": PlotRatio,
                                                                "SVD": PlotSvdStudy,
                                                                "SVDCut": PlotSvdCut
                                                            }
                                                        )

PlotParamsAccordion     = widgets.Accordion(children=[widgets.VBox([PlotControls, PlotInteractive])])
PlotParamsAccordion.set_title(0, "Plot Parameters")

''' ----- Arguments portion -----'''
PlotArgsNN              = widgets.Text(value='[3]', description='number of exponentials in single nucleon to sweep over', style=style, layout=layout)
PlotArgsNNEL            = widgets.Text(value='[0]', description='number of elastic e.s. to try', style=style, layout=layout)
PlotArgsGEVP0            = widgets.Text(value='[4,4,5,5,6,6]', description='list of GEVP times t0', style=style, layout=layout)
PlotArgsGEVPd            = widgets.Text(value='[8,10,10,12,10,12]', description='list of GEVP times td', style=style, layout=layout)
PlotArgsTmin            = widgets.Text(value='[2,3,4,5,6,7,8]', description='values of t_min in NN fit', style=style, layout=layout)
PlotArgsFig             = widgets.Dropdown(options=['pdf','png'], description=' Save fig as: ', style=style, layout=layout)
PlotArgsRatio           = widgets.Checkbox(value=False, description='fit from RATIO correlator? ', style=style, layout=layout)
PlotArgsEvp             = widgets.Checkbox(value=True, description='load evp (vs gevp) data? ', style=style, layout=layout)
PlotArgsGSCons          = widgets.Checkbox(value=False, description='use gs only conspiracy model? ', style=style, layout=layout)

PlotArgsControls        = widgets.VBox([PlotArgsNN,PlotArgsNNEL,PlotArgsGEVP0,PlotArgsGEVPd,PlotArgsTmin,PlotArgsFig,PlotArgsRatio,PlotArgsEvp,PlotArgsGSCons])

PlotArgsInteractive     = widgets.interactive_output(PlotMakeArgs,
                                                            {   
                                                                "PlotnNList": PlotArgsNN,
                                                                "PlotnnElList": PlotArgsNNEL,
                                                                "PlotGEVP0List": PlotArgsGEVP0,
                                                                "PlotGEVPdList": PlotArgsGEVPd,
                                                                "PlottminList": PlotArgsTmin,
                                                                "plotfigtype": PlotArgsFig,
                                                                "plotratio": PlotArgsRatio,
                                                                "plotevp": PlotArgsEvp,
                                                                "plotgscons": PlotArgsGSCons
                                                            }
                                                        )

PlotArgsAccordion       = widgets.Accordion(children=[widgets.VBox([PlotArgsControls, PlotArgsInteractive])])
PlotArgsAccordion.set_title(0, " Plot Args")

''' ----- Plot Limits Portion -----'''
Plot0UpLim              = widgets.FloatText(value=0, description='Top plot upper limit', style=style, layout=layout)
Plot0LowLim             = widgets.FloatText(value=0, description='Top plot lower limit', style=style, layout=layout)
Plot1UpLim              = widgets.FloatText(value=0, description='Middle plot upper limit', style=style, layout=layout)
Plot1LowLim             = widgets.FloatText(value=0, description='Middle plot lower limit', style=style, layout=layout)

PlotLimitsAccordion     = widgets.Accordion(children=[widgets.VBox([Plot0UpLim,Plot0LowLim,Plot1UpLim,Plot1LowLim])]) 
PlotLimitsAccordion.set_title(0, "Plot Limits")

PlotAccordionStack   = widgets.VBox([PlotParamsAccordion, PlotArgsAccordion, PlotLimitsAccordion])

PlotButtonOut = widgets.Output()

In [13]:
PlotButton           = widgets.Button(description='Plot')

PlotButton.on_click(PlottingButton)

OnePlotTextHeader    = widgets.HTML(f"<b>Display 1 specific plot </b>")

display(OnePlotTextHeader, PlotAccordionStack, PlotButton, PlotButtonOut)

HTML(value='<b>Display 1 specific plot </b>')

Button(description='Plot', style=ButtonStyle())

Output()

### Summary Plot

In [14]:
''' Interactive Functions. Feel free to change where needed. Keep collapsed to reduce clutter. '''
import plot_nn_stability_gevp as plotgevp

def SummaryPlot(filename,lims,SummaryPlotnNList,SummaryPlotnnElList,SummaryPlotGEVP0List,SummaryPlotGEVPdList,SummaryPlottminList,SummaryPlotfigtype,SummaryPlotratio,SummaryPlotevp,SummaryPlotgscons):
    argv = [
        "plot_nn_stability_gevp.py",
        filename
    ] 
    argv.append("--summary")
    SummaryGEVPt0       = WidgetTextToArray(SummaryPlotGEVP0List)
    SummaryGEVPtd       = WidgetTextToArray(SummaryPlotGEVPdList)
    SummaryPlotGEVPList= []
    for i in range(len(SummaryGEVPt0)):
        SummaryPlotGEVPList.append(f'{SummaryGEVPt0[i]}-{SummaryGEVPtd[i]}')
    argv += ["--n_N"] + [str(x) for x in WidgetTextToArray(SummaryPlotnNList)]
    argv += ["--nn_el"] + [str(x) for x in WidgetTextToArray(SummaryPlotnnElList)]
    argv += ["--gevp"] + [str(x) for x in SummaryPlotGEVPList]
    argv += ["--tmin"] + [str(x) for x in WidgetTextToArray(SummaryPlottminList)]
    argv += ["--fig_type", SummaryPlotfigtype]
    if SummaryPlotratio:
        argv.append("--ratio")
    if not SummaryPlotevp:
        argv.append("--evp")
    if SummaryPlotgscons:
        argv.append("--gs_cons")
    sys.argv = argv
    allirreps = []
    plt.close('all')
    plotgevp.main(params=allirreps)
    clear_output()
    figs = [plt.figure(n) for n in plt.get_fignums()]
    fig1, fig2, fig3, fig4 = figs
    Plot1Ax = fig1.axes[0]
    Plot2Ax = fig2.axes[0]
    Plot3Ax = fig3.axes[0]
    Plot4Ax = fig4.axes[0]
    if (lims[0]-lims[1])!=0:
        Plot1Ax.set_ylim(lims[1],lims[0])
    if (lims[2]-lims[3])!=0:
        Plot2Ax.set_ylim(lims[3],lims[2])
    if (lims[4]-lims[5])!=0:
        Plot3Ax.set_ylim(lims[5],lims[4])
    if (lims[6]-lims[7])!=0:
        Plot4Ax.set_ylim(lims[7],lims[6])
    plt.show(fig1)
    plt.show(fig2)
    plt.show(fig3)
    plt.show(fig4)

def SummaryPlotMakeFilename(GEVP, TNorm, NRange, RRange, NStates, AgCons, NElastic, Block, Ratio, SVD, SVDCut):
    nn_iso  = 'deuteron'
    irrep   = ('0','T1g',0)
    RepPath = f"{irrep[0]}{irrep[1]}{irrep[2]}"
    ft0td   = f'{GEVP[0]}-{GEVP[1]}'
    fN      = f'{NRange[0]}-{NRange[1]}'
    fagcons = AgCons
    fnN     = NStates
    fe      = NElastic
    fR      = f'{RRange[0]}-{RRange[1]}'
    fRatio  = Ratio
    fBlock  = Block
    ftnorm  = TNorm
    if SVD:
        fSVDCut = SVDCut
        filename = f"result/{RepPath}/NN_{nn_iso}_tnorm{ftnorm}_t0-td_{ft0td}_N_n{fnN}_t_{fN}_NN_{fagcons}_e{fe}_t_{fR}_ratio_{fRatio}_block{fBlock}_svdcut{fSVDCut}.pickle"
    else:
        filename = f"result/{RepPath}/NN_{nn_iso}_tnorm{ftnorm}_t0-td_{ft0td}_N_n{fnN}_t_{fN}_NN_{fagcons}_e{fe}_t_{fR}_ratio_{fRatio}_block{fBlock}.pickle"
    return filename

def SummaryPlottingButton(b):
    with SummaryPlotButtonOut:
        plt.close('all')
        clear_output()
        SummaryPlotFileName= SummaryPlotMakeFilename(GEVP=SummaryPlotGEVPt0td.value, 
                                   TNorm=SummaryPlotTNorm.value, NRange=SummaryPlotNRange.value, RRange=SummaryPlotRRange.value, 
                                   NStates=SummaryPlotNStates.value, AgCons=SummaryPlotAgCons.value, 
                                   NElastic=SummaryPlotNElastic.value, Block=SummaryPlotBlock.value, Ratio=SummaryPlotRatio.value, SVD=SummaryPlotSvdStudy.value, SVDCut=SummaryPlotSvdCut.value)
        limits      = [SummaryPlot1UpLim.value,SummaryPlot1LowLim.value,SummaryPlot2UpLim.value,SummaryPlot2LowLim.value,SummaryPlot3UpLim.value,SummaryPlot3LowLim.value,SummaryPlot4UpLim.value,SummaryPlot4LowLim.value]
        SummaryPlot(filename=SummaryPlotFileName,lims=limits,SummaryPlotnNList=SummaryPlotArgsNN,SummaryPlotnnElList=SummaryPlotArgsNNEL,SummaryPlotGEVP0List=SummaryPlotArgsGEVP0,SummaryPlotGEVPdList=SummaryPlotArgsGEVPd,SummaryPlottminList=SummaryPlotArgsTmin,SummaryPlotfigtype=SummaryPlotArgsFig.value,SummaryPlotratio=SummaryPlotArgsRatio.value,SummaryPlotevp=SummaryPlotArgsEvp.value,SummaryPlotgscons=SummaryPlotArgsGSCons.value)

def PlotMakeArgs(SummaryPlotnNList,SummaryPlotnnElList,SummaryPlotGEVP0List,SummaryPlotGEVPdList,SummaryPlottminList,SummaryPlotfigtype,SummaryPlotratio,SummaryPlotevp,SummaryPlotgscons):
    return

def SummaryPlotUpdateMain(GEVP,Block,NStates,NRange,RRange,TNorm,AgCons,NElastic,Ratio,Evp,SVD,SVDCut):
    return


In [15]:
''' Interactive Widgets. Feel free to change where needed. Keep collapsed to reduce clutter. '''

''' ----- Main Portion ----- '''
SummaryPlotGEVPt0td              = widgets.IntRangeSlider(value=[3,8], min=0, max=12, step=1, description='GEVP t0-td: ', style=style, layout=layout)
SummaryPlotBlock                 = widgets.IntSlider(value=2, min=0, max=10, step=1, description='Block: ', style=style, layout=layout)
SummaryPlotNStates               = widgets.IntSlider(value=3, min=0, max=5, step=1, description='Nstates: ', style=style, layout=layout)
SummaryPlotRRange                = widgets.IntRangeSlider(value=[3,15], min=0, max=15, step=1, description='R Range t0-td: ', style=style, layout=layout)
SummaryPlotNRange                = widgets.IntRangeSlider(value=[3,20], min=0, max=20, step=1, description='N Range t0-td: ', style=style, layout=layout)
SummaryPlotTNorm                 = widgets.IntSlider(value=3, min=0, max=5, step=1, description='tNorm: ', style=style, layout=layout)
SummaryPlotNElastic              = widgets.IntSlider(value=0, min=0, max=5, step=1, description='R N Elastic states: ', style=style, layout=layout)
SummaryPlotAgCons                = widgets.Dropdown(options=['conspire', 'agnostic'], description='Agnostic or Conspiracy? ', orientation='horizontal', style=style, layout=layout)
SummaryPlotEvpGevp               = widgets.Dropdown(options=['evp','gevp'], description='evp or gevp', orientation='horizontal', style=style, layout=layout)
SummaryPlotRatio                 = widgets.Checkbox(value=False, description='Ratio? ', style=style, layout=layout)
SummaryPlotSvdStudy              = widgets.Checkbox(value=False, description='Study SVD? ', style=style, layout=layout)
SummaryPlotSvdCut                = widgets.Text(value='1e-08', description='SVD Cut', style=style, layout=layout)

SummaryPlotMainControls         = widgets.VBox([SummaryPlotGEVPt0td,SummaryPlotNRange,SummaryPlotRRange,SummaryPlotTNorm, SummaryPlotNStates,SummaryPlotNElastic,SummaryPlotAgCons, SummaryPlotEvpGevp, SummaryPlotRatio, SummaryPlotSvdStudy, SummaryPlotSvdCut])

SummaryPlotMainInteractive = widgets.interactive_output(SummaryPlotUpdateMain,
                                                            {   
                                                                "GEVP": SummaryPlotGEVPt0td,
                                                                "Block": SummaryPlotBlock,
                                                                "NStates": SummaryPlotNStates,
                                                                "NRange": SummaryPlotNRange,
                                                                "RRange": SummaryPlotRRange,
                                                                "TNorm": SummaryPlotTNorm,
                                                                "AgCons": SummaryPlotAgCons,
                                                                "NElastic": SummaryPlotNElastic,
                                                                "Ratio": SummaryPlotRatio,
                                                                "Evp": SummaryPlotEvpGevp,
                                                                "SVD": PlotSvdStudy,
                                                                "SVDCut": PlotSvdCut
                                                            }
                                                        )

SummaryPlotMainParamsAccordion      = widgets.Accordion(children=[widgets.VBox([SummaryPlotMainControls, SummaryPlotMainInteractive])])
SummaryPlotMainParamsAccordion.set_title(0, "Summary Plot Parameters")

SummaryPlot1UpLim              = widgets.FloatText(value=0, description='GEVP plot upper limit', style=style, layout=layout)
SummaryPlot1LowLim             = widgets.FloatText(value=0, description='GEVP plot lower limit', style=style, layout=layout)
SummaryPlot2UpLim              = widgets.FloatText(value=0, description='GEVP dElab plot upper limit', style=style, layout=layout)
SummaryPlot2LowLim             = widgets.FloatText(value=0, description='GEVP dElab plot lower limit', style=style, layout=layout)
SummaryPlot3UpLim              = widgets.FloatText(value=0, description='Tmin plot upper limit', style=style, layout=layout)
SummaryPlot3LowLim             = widgets.FloatText(value=0, description='Tmin plot lower limit', style=style, layout=layout)
SummaryPlot4UpLim              = widgets.FloatText(value=0, description='Tmin dElab plot upper limit', style=style, layout=layout)
SummaryPlot4LowLim             = widgets.FloatText(value=0, description='Tmin dElab plot lower limit', style=style, layout=layout)

SummaryPlotLimitsAccordion     = widgets.Accordion(children=[widgets.VBox([SummaryPlot1UpLim,SummaryPlot1LowLim,SummaryPlot2UpLim,SummaryPlot2LowLim,SummaryPlot3UpLim,SummaryPlot3LowLim,SummaryPlot4UpLim,SummaryPlot4LowLim])]) 
SummaryPlotLimitsAccordion.set_title(0, "Summary Plot Limits")

''' ----- Summary Plot Args -----'''
SummaryPlotArgsNN              = widgets.Text(value='[3]', description='number of exponentials in single nucleon to sweep over', style=style, layout=layout)
SummaryPlotArgsNNEL            = widgets.Text(value='[0]', description='number of elastic e.s. to try', style=style, layout=layout)
SummaryPlotArgsGEVP0            = widgets.Text(value='[4,4,5,5,6,6]', description='list of GEVP times t0', style=style, layout=layout)
SummaryPlotArgsGEVPd            = widgets.Text(value='[8,10,10,12,10,12]', description='list of GEVP times td', style=style, layout=layout)
SummaryPlotArgsTmin            = widgets.Text(value='[2,3,4,5,6,7,8]', description='values of t_min in NN fit', style=style, layout=layout)
SummaryPlotArgsFig             = widgets.Dropdown(options=['pdf','png'], description=' Save fig as: ', style=style, layout=layout)
SummaryPlotArgsRatio           = widgets.Checkbox(value=False, description='fit from RATIO correlator? ', style=style, layout=layout)
SummaryPlotArgsEvp             = widgets.Checkbox(value=True, description='load evp (vs gevp) data? ', style=style, layout=layout)
SummaryPlotArgsGSCons          = widgets.Checkbox(value=False, description='use gs only conspiracy model? ', style=style, layout=layout)

SummaryPlotArgsControls        = widgets.VBox([SummaryPlotArgsNN,SummaryPlotArgsNNEL,SummaryPlotArgsGEVP0,SummaryPlotArgsGEVPd,SummaryPlotArgsTmin,SummaryPlotArgsFig,SummaryPlotArgsRatio,SummaryPlotArgsEvp,SummaryPlotArgsGSCons])

SummaryPlotArgsInteractive     = widgets.interactive_output(PlotMakeArgs,
                                                            {   
                                                                "SummaryPlotnNList": SummaryPlotArgsNN,
                                                                "SummaryPlotnnElList": SummaryPlotArgsNNEL,
                                                                "SummaryPlotGEVP0List": SummaryPlotArgsGEVP0,
                                                                "SummaryPlotGEVPdList": SummaryPlotArgsGEVPd,
                                                                "SummaryPlottminList": SummaryPlotArgsTmin,
                                                                "SummaryPlotfigtype": SummaryPlotArgsFig,
                                                                "SummaryPlotratio": SummaryPlotArgsRatio,
                                                                "SummaryPlotevp": SummaryPlotArgsEvp,
                                                                "SummaryPlotgscons": SummaryPlotArgsGSCons
                                                            }
                                                        )

SummaryPlotArgsAccordion      = widgets.Accordion(children=[widgets.VBox([SummaryPlotArgsControls, SummaryPlotArgsInteractive])])
SummaryPlotArgsAccordion.set_title(0, "Summary Plot Args Parameters")

SummaryPlotAccordionStack   = widgets.VBox([SummaryPlotMainParamsAccordion, SummaryPlotArgsAccordion, SummaryPlotLimitsAccordion])

SummaryPlotButtonOut = widgets.Output()

In [16]:
SummaryPlotButton           = widgets.Button(description='Display Summary Plot')

SummaryPlotButton.on_click(SummaryPlottingButton)

SummaryPlotTextHeader    = widgets.HTML(f"<b>Summary Plot </b>")

display(SummaryPlotTextHeader, SummaryPlotAccordionStack, SummaryPlotButton, SummaryPlotButtonOut)

HTML(value='<b>Summary Plot </b>')

Button(description='Display Summary Plot', style=ButtonStyle())

Output()

# Fit Parameters & Quality of Fit Metrics

### Plain Text

In [17]:
''' Interactive Functions. Feel free to change where needed. Keep collapsed to reduce clutter. '''

def PrintParameters(filename, irrep):

    data = gv.load(filename)

    IrrepData   = {k:v
                   for k,v in data.items()
                   if k[0][0] == irrep[0]}
    e_values    = {k:v
                   for k,v in IrrepData.items()
                   if  k[1].startswith('e')}
    z_values    = {k:v
                   for k,v in IrrepData.items()
                   if  k[1].startswith('z')}
    
    with PrintSingleNuke:
        clear_output()
        print(irrep)
        en_values   = {k:v
                       for k,v in e_values.items()
                       if k[0][1] == 'N'}
        zn_values   = {k:v
                       for k,v in z_values.items()
                       if k[0][1] == 'N'}
        for (ke,ve), (kz,vz) in zip(en_values.items(), zn_values.items()):
            print(f'{ke[1]}={ve}; p={ke[0][2]}')
            print(f'{kz[1]}={vz}; p={kz[0][2]}')
    with PrintDoubleNuke:
        clear_output()
        er_values   = {k:v
                       for k,v in e_values.items()
                       if k[0][1] == 'R'}
        zr_values   = {k:v
                       for k,v in z_values.items()
                       if k[0][1] == 'R'}
        for (ke,ve), (kz,vz) in zip(er_values.items(), zr_values.items()):
            print(f'{ke[1]}={ve}; p={ke[0][2]}')
            print(f'{kz[1]}={vz}; p={kz[0][2]}')
    with PrintQuality:
        clear_output()
        quality_vals= {k:v
                   for k,v in IrrepData.items()
                   if len(k[0]) == 1}
        q_val       = quality_vals[(irrep[0],),'Q']
        chi2dof     = quality_vals[(irrep[0],),'chi2']/quality_vals[(irrep[0],),'dof']
        loggbf      = quality_vals[(irrep[0],),'logGBF']
        print(f'Q: {q_val}')
        print(f'chi2/dof: {chi2dof}')
        print(f'logGBF: {loggbf}')
    with PrintSettings:
        clear_output()
        print(f'svdcut/svdn: {IrrepData[(irrep[0],),'svdcut']}/{IrrepData[(irrep[0],),'svdn']}')
        print('Not quite sure where this egs term goes')
        print(f'egs: {data[irrep[0],'egs']}')
    display(PrintParamsAccordion)

def MakeFilename(Irrep, GEVP, TNorm, NRange, RRange, NStates, AgCons, NElastic, Block, Ratio):
    nn_iso  = 'deuteron'
    irrep   = Irrep[0]
    RepPath = f"{irrep[0]}{irrep[1]}{irrep[2]}"
    ft0td   = f'{GEVP[0]}-{GEVP[1]}'
    fN      = f'{NRange[0]}-{NRange[1]}'
    fagcons = AgCons
    fnN     = NStates
    fe      = NElastic
    fR      = f'{RRange[0]}-{RRange[1]}'
    fRatio  = Ratio
    fBlock  = Block
    ftnorm  = TNorm
    filename = f"result/{RepPath}/NN_{nn_iso}_tnorm{ftnorm}_t0-td_{ft0td}_N_n{fnN}_t_{fN}_NN_{fagcons}_e{fe}_t_{fR}_ratio_{fRatio}_block{fBlock}.pickle"
    return filename

def PrintButton(b):
    with PrintButtonOut:
        clear_output()
        FitFileName = MakeFilename(Irrep=ChooseIrrep.value, GEVP=ChooseGEVPt0td.value, 
                                   TNorm=ChooseTNorm.value, NRange=ChooseNRange.value, RRange=ChooseRRange.value, 
                                   NStates=ChooseNStates.value, AgCons=ChooseAgCons.value, 
                                   NElastic=ChooseNElastic.value, Block=ChooseBlock.value, Ratio=ChooseRatio.value)
        PrintParameters(filename=FitFileName,irrep=ChooseIrrep.value)

def update(Irrep, GEVP, TNorm, NRange, RRange, NStates, AgCons, NElastic, Block, Ratio):
    return

In [18]:
''' Interactive Widgets. Feel free to change where needed. Keep collapsed to reduce clutter. '''
ChooseIrrep                 = widgets.Dropdown(options=DeuteronIrreps, description='Choose Irrep: ', style=style, layout=layout)
ChooseGEVPt0td              = widgets.IntRangeSlider(value=[3,8], min=0, max=12, step=1, description='GEVP t0-td: ', style=style, layout=layout)
ChooseTNorm                 = widgets.IntSlider(value=3, min=0, max=5, step=1, description='tNorm: ', style=style, layout=layout)
ChooseNElastic              = widgets.IntSlider(value=0, min=0, max=5, step=1, description='R N Elastic states: ', style=style, layout=layout)
ChooseBlock                 = widgets.IntSlider(value=2, min=0, max=10, step=1, description='Block: ', style=style, layout=layout)
ChooseNStates               = widgets.IntSlider(value=3, min=0, max=5, step=1, description='Nstates: ', style=style, layout=layout)
ChooseNRange                = widgets.IntRangeSlider(value=[3,20], min=0, max=20, step=1, description='N Range t0-td: ', style=style, layout=layout)
ChooseRRange                = widgets.IntRangeSlider(value=[3,15], min=0, max=15, step=1, description='R Range t0-td: ', style=style, layout=layout)
ChooseAgCons                = widgets.Dropdown(options=['conspire', 'agnostic'], description='Agnostic or Conspiracy? ', orientation='horizontal', style=style, layout=layout)

ChooseRatio                 = widgets.Checkbox(value=False, description='Ratio? ', style=style, layout=layout)

controls                    = widgets.VBox([ChooseIrrep, ChooseGEVPt0td, ChooseTNorm, ChooseNStates, ChooseNElastic, ChooseNRange, ChooseRRange, ChooseBlock, ChooseAgCons, ChooseRatio])

PrintInteractive            = widgets.interactive_output(update,
                                                            {   
                                                                "Irrep": ChooseIrrep,
                                                                "GEVP": ChooseGEVPt0td,
                                                                "TNorm": ChooseTNorm,
                                                                "NRange": ChooseNRange,
                                                                "RRange": ChooseRRange,
                                                                "NStates": ChooseNStates,
                                                                "AgCons": ChooseAgCons,
                                                                "NElastic": ChooseNElastic,
                                                                "Block": ChooseBlock,
                                                                "Ratio": ChooseRatio,
                                                            }
                                                        )

ChoosePrintParamsAccordion  = widgets.Accordion(children=[widgets.VBox([controls, PrintInteractive])])
ChoosePrintParamsAccordion.set_title(0, "Fit Parameters")

''' Print Parameters in 4 Accordions '''

PrintSingleNuke             = widgets.Output()
PrintDoubleNuke             = widgets.Output()
PrintQuality                = widgets.Output()
PrintSettings               = widgets.Output()

PrintParamsAccordion        = widgets.Accordion(children=[PrintSingleNuke, PrintDoubleNuke, PrintQuality, PrintSettings])
PrintParamsAccordion.set_title(0, "Single Nucleon Parameters")
PrintParamsAccordion.set_title(1, "Double Nucleon Parameters")
PrintParamsAccordion.set_title(2, "Quality of Fit Metrics")
PrintParamsAccordion.set_title(3, "Settings")

PrintButtonOut              = widgets.Output()

In [19]:
''' Actual Parameter Printing '''
PrintParamsButton           = widgets.Button(description='Display Plain Text')

PrintParamsButton.on_click(PrintButton)

display(ChoosePrintParamsAccordion, PrintParamsButton, PrintButtonOut)

Accordion(children=(VBox(children=(VBox(children=(Dropdown(description='Choose Irrep: ', layout=Layout(width='…

Button(description='Display Plain Text', style=ButtonStyle())

Output()